# Incremental / CDC-Style Processing

This notebook demonstrates incremental processing for the healthcare
data engineering pipeline.

Instead of processing the complete dataset during every execution,
the pipeline identifies new or changed records using the `updated_at`
timestamp.

A watermark stores the latest successfully processed timestamp.

The incremental workflow is:

OLTP Source
    ↓
updated_at timestamp
    ↓
Watermark
    ↓
Identify new/changed records
    ↓
Process only incremental records
    ↓
Update watermark

This CDC-style approach reduces unnecessary processing and supports
efficient data pipeline execution.

## Step 1 — Read the Processing Watermark

The watermark stores the latest `updated_at` timestamp processed by
the previous pipeline execution.

Only records with an `updated_at` value greater than this watermark
will be considered for incremental processing.

In [0]:
from pyspark.sql import functions as F

appointments = spark.table("silver_appointments")

max_updated_at = (
    appointments
    .select(
        F.max("updated_at").alias("max_updated_at")
    )
    .collect()[0]["max_updated_at"]
)

print("Latest appointment updated_at:", max_updated_at)

Latest appointment updated_at: 2026-08-19 01:50:09


In [0]:
watermark_data = [
    ("appointments", max_updated_at)
]

watermark_df = spark.createDataFrame(
    watermark_data,
    ["table_name", "last_processed_timestamp"]
)

watermark_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("etl_watermark")

print("Watermark table created successfully.")

Watermark table created successfully.


In [0]:
display(spark.table("etl_watermark"))

table_name,last_processed_timestamp
appointments,2026-08-19T01:50:09.000Z


## Step 2 — Detect New or Changed Records

The pipeline compares the `updated_at` timestamp of incoming
appointment records with the previously stored watermark.

Only records with:

`updated_at > last_processed_timestamp`

are considered new or changed records.

This prevents the pipeline from unnecessarily processing the
complete appointment dataset during every execution.

In [0]:
appointments = spark.table("silver_appointments")

last_processed_timestamp = (
    spark.table("etl_watermark")
    .filter(F.col("table_name") == "appointments")
    .select("last_processed_timestamp")
    .collect()[0]["last_processed_timestamp"]
)

incremental_appointments = appointments.filter(
    F.col("updated_at") > F.lit(last_processed_timestamp)
)

print(
    "Last processed timestamp:",
    last_processed_timestamp
)

print(
    "Total appointments:",
    appointments.count()
)

print(
    "Incremental appointments:",
    incremental_appointments.count()
)

display(incremental_appointments)

Last processed timestamp: 2026-08-19 01:50:09
Total appointments: 500
Incremental appointments: 0


appointment_id,patient_id,doctor_id,department_id,appointment_date,appointment_status,visit_type,reason,wait_time_minutes,consultation_duration_minutes,created_at,updated_at


## Step 3 — Simulate an Incoming Change

To demonstrate incremental processing, one existing appointment is
used to create a controlled incoming change.

The appointment's `reason` is changed and its `updated_at` timestamp
is set to a value newer than the stored watermark.

This is only a CDC demonstration. The original OLTP and Silver
records are not modified.

In [0]:
# Select one existing appointment as a demonstration record
demo_appointment = (
    appointments
    .filter(F.col("appointment_id") == 1)
    .limit(1)
)

# Create a simulated changed version
incoming_change = (
    demo_appointment
    .withColumn(
        "reason",
        F.lit("Follow-up consultation - CDC demonstration")
    )
    .withColumn(
        "updated_at",
        F.current_timestamp()
    )
)

display(incoming_change)

appointment_id,patient_id,doctor_id,department_id,appointment_date,appointment_status,visit_type,reason,wait_time_minutes,consultation_duration_minutes,created_at,updated_at
1,1,1,1,2025-03-22,SCHEDULED,EMERGENCY,Follow-up consultation - CDC demonstration,52,24,2025-04-04T23:53:17.000Z,2026-09-05T11:41:40.299Z


In [0]:
incremental_change = incoming_change.filter(
    F.col("updated_at") > F.lit(last_processed_timestamp)
)

print(
    "Records detected for incremental processing:",
    incremental_change.count()
)

display(incremental_change)

Records detected for incremental processing: 1


## Step 4 — Update the Processing Watermark

After incremental records are successfully processed, the watermark
is updated to the latest `updated_at` timestamp that was processed.

This ensures that the same records are not processed again during
the next pipeline execution.

In [0]:
new_watermark = (
    incremental_change
    .select(
        F.max("updated_at").alias("max_updated_at")
    )
    .collect()[0]["max_updated_at"]
)

print("New watermark:", new_watermark)

New watermark: 2026-09-05 11:45:30.523829


In [0]:
updated_watermark = spark.createDataFrame(
    [
        ("appointments", new_watermark)
    ],
    ["table_name", "last_processed_timestamp"]
)

updated_watermark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("etl_watermark")

print("Watermark updated successfully.")

Watermark updated successfully.


In [0]:
display(spark.table("etl_watermark"))

table_name,last_processed_timestamp
appointments,2026-09-05T11:45:30.523Z


In [0]:
current_watermark = (
    spark.table("etl_watermark")
    .filter(F.col("table_name") == "appointments")
    .select("last_processed_timestamp")
    .collect()[0]["last_processed_timestamp"]
)

remaining_incremental = appointments.filter(
    F.col("updated_at") > F.lit(current_watermark)
)

print(
    "Records remaining for incremental processing:",
    remaining_incremental.count()
)

Records remaining for incremental processing: 0
